# ANN Lab Task - Linear Regression
### Dataset 1: California Housing
### Dataset 2: Student Performance

In [ ]:
!pip install gradio flask -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

## Dataset 1 - California Housing

In [ ]:
df1 = pd.read_csv(r"C:\Users\Geo Computer\OneDrive\Desktop\New folder\california_housing.csv")
df1.head()

In [ ]:
df1.shape

In [ ]:
df1.describe()

In [ ]:
df1.isnull().sum()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df1.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap - California Housing')
plt.show()

In [ ]:
plt.scatter(df1['MedInc'], df1['MedHouseValue'], color='blue', alpha=0.5)
plt.xlabel('Median Income')
plt.ylabel('House Value')
plt.title('Income vs House Value')
plt.show()

In [ ]:
X1 = df1.drop('MedHouseValue', axis=1)
y1 = df1['MedHouseValue']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

scaler1 = StandardScaler()
X1_train = scaler1.fit_transform(X1_train)
X1_test  = scaler1.transform(X1_test)

In [ ]:
model1 = LinearRegression()
model1.fit(X1_train, y1_train)

In [ ]:
y1_pred = model1.predict(X1_test)

print('MSE  :', mean_squared_error(y1_test, y1_pred))
print('RMSE :', np.sqrt(mean_squared_error(y1_test, y1_pred)))
print('MAE  :', mean_absolute_error(y1_test, y1_pred))
print('R2   :', r2_score(y1_test, y1_pred))

In [ ]:
plt.scatter(y1_test, y1_pred, alpha=0.6, color='steelblue')
plt.plot([y1_test.min(), y1_test.max()], [y1_test.min(), y1_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted - Housing')
plt.show()

## Dataset 2 - Student Performance

In [ ]:
df2 = pd.read_csv(r"C:\Users\Geo Computer\OneDrive\Desktop\New folder\student_performance.csv")
df2.head()

In [ ]:
df2.shape

In [ ]:
df2.describe()

In [ ]:
df2.isnull().sum()

In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(df2.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap - Student Performance')
plt.show()

In [ ]:
plt.scatter(df2['StudyHours'], df2['ExamScore'], color='green', alpha=0.6)
plt.xlabel('Study Hours')
plt.ylabel('Exam Score')
plt.title('Study Hours vs Exam Score')
plt.show()

In [ ]:
X2 = df2.drop('ExamScore', axis=1)
y2 = df2['ExamScore']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train)
X2_test  = scaler2.transform(X2_test)

In [ ]:
model2 = LinearRegression()
model2.fit(X2_train, y2_train)

In [ ]:
y2_pred = model2.predict(X2_test)

print('MSE  :', mean_squared_error(y2_test, y2_pred))
print('RMSE :', np.sqrt(mean_squared_error(y2_test, y2_pred)))
print('MAE  :', mean_absolute_error(y2_test, y2_pred))
print('R2   :', r2_score(y2_test, y2_pred))

In [ ]:
plt.scatter(y2_test, y2_pred, alpha=0.6, color='green')
plt.plot([y2_test.min(), y2_test.max()], [y2_test.min(), y2_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted - Student')
plt.show()

## Gradio Interface

In [ ]:
import gradio as gr

def housing_predict(MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude):
    data = np.array([[MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude]])
    data = scaler1.transform(data)
    result = model1.predict(data)[0]
    return f"Predicted House Value: ${result * 100000:,.0f}"

def student_predict(StudyHours, PreviousScore, SleepHours, AttendancePct, PracticeTests):
    data = np.array([[StudyHours, PreviousScore, SleepHours, AttendancePct, PracticeTests]])
    data = scaler2.transform(data)
    result = model2.predict(data)[0]
    result = np.clip(result, 0, 100)
    return f"Predicted Exam Score: {result:.2f} / 100"


with gr.Blocks(title="Linear Regression") as app:
    gr.Markdown("# Linear Regression - ANN Lab")

    with gr.Tab("Housing Price"):
        gr.Markdown("### California Housing Dataset")
        with gr.Row():
            inc  = gr.Slider(0.5, 15,    value=5,    label="Median Income")
            age  = gr.Slider(1,   52,    value=20,   label="House Age")
            rooms= gr.Slider(1,   15,    value=5,    label="Avg Rooms")
            beds = gr.Slider(0.5, 5,     value=1,    label="Avg Bedrooms")
        with gr.Row():
            pop  = gr.Slider(50,  35000, value=1500, label="Population")
            occ  = gr.Slider(1,   10,    value=3,    label="Avg Occupancy")
            lat  = gr.Slider(32,  42,    value=36,   label="Latitude")
            lon  = gr.Slider(-124,-114,  value=-120, label="Longitude")
        btn1 = gr.Button("Predict")
        out1 = gr.Textbox(label="Result")
        btn1.click(housing_predict, inputs=[inc,age,rooms,beds,pop,occ,lat,lon], outputs=out1)

    with gr.Tab("Student Score"):
        gr.Markdown("### Student Performance Dataset")
        sh  = gr.Slider(1,  10,  value=5,  label="Study Hours")
        ps  = gr.Slider(40, 95,  value=70, label="Previous Score")
        slp = gr.Slider(4,  10,  value=7,  label="Sleep Hours")
        att = gr.Slider(50, 100, value=80, label="Attendance %")
        pt  = gr.Slider(0,  10,  value=3,  label="Practice Tests")
        btn2 = gr.Button("Predict")
        out2 = gr.Textbox(label="Result")
        btn2.click(student_predict, inputs=[sh,ps,slp,att,pt], outputs=out2)

app.launch(inline=True)

## Flask Interface

In [ ]:
from flask import Flask, request, jsonify, render_template_string
import threading

flask_app = Flask(__name__)

page = '''
<!DOCTYPE html>
<html>
<head>
  <title>Linear Regression</title>
  <style>
    body { font-family: Arial; background: #1e1e2e; color: #cdd6f4; padding: 30px; }
    h1   { color: #cba6f7; text-align: center; }
    h3   { color: #89b4fa; }
    .box {
      background: #313244; border-radius: 12px;
      padding: 25px; margin: 20px auto;
      max-width: 550px;
    }
    label  { display: block; margin-top: 10px; font-size: 14px; }
    input  { width: 100%; padding: 8px; margin-top: 4px; border-radius: 6px; border: none; background: #45475a; color: white; }
    button { margin-top: 15px; width: 100%; padding: 10px; background: #cba6f7; color: #1e1e2e; font-weight: bold; border: none; border-radius: 8px; cursor: pointer; }
    .result { margin-top: 12px; padding: 12px; background: #a6e3a1; color: #1e1e2e; border-radius: 8px; font-weight: bold; display:none; }
  </style>
</head>
<body>
  <h1>Linear Regression - ANN Lab</h1>

  <div class="box">
    <h3>Dataset 1 - California Housing</h3>
    <label>Median Income</label>    <input id="inc"   type="number" value="5" step="0.1">
    <label>House Age</label>        <input id="age"   type="number" value="20">
    <label>Avg Rooms</label>        <input id="rooms" type="number" value="5" step="0.1">
    <label>Avg Bedrooms</label>     <input id="beds"  type="number" value="1" step="0.05">
    <label>Population</label>       <input id="pop"   type="number" value="1500">
    <label>Avg Occupancy</label>    <input id="occ"   type="number" value="3" step="0.1">
    <label>Latitude</label>         <input id="lat"   type="number" value="36" step="0.1">
    <label>Longitude</label>        <input id="lon"   type="number" value="-120" step="0.1">
    <button onclick="predictHousing()">Predict House Price</button>
    <div class="result" id="h_out"></div>
  </div>

  <div class="box">
    <h3>Dataset 2 - Student Performance</h3>
    <label>Study Hours</label>    <input id="sh"  type="number" value="5" step="0.5">
    <label>Previous Score</label> <input id="ps"  type="number" value="70">
    <label>Sleep Hours</label>    <input id="slp" type="number" value="7" step="0.5">
    <label>Attendance %</label>   <input id="att" type="number" value="80">
    <label>Practice Tests</label> <input id="pt"  type="number" value="3">
    <button onclick="predictStudent()">Predict Exam Score</button>
    <div class="result" id="s_out"></div>
  </div>

  <script>
    async function predictHousing() {
      const res = await fetch('/predict/housing', {
        method: 'POST', headers: {'Content-Type': 'application/json'},
        body: JSON.stringify({
          MedInc: +inc.value, HouseAge: +age.value, AveRooms: +rooms.value,
          AveBedrms: +beds.value, Population: +pop.value, AveOccup: +occ.value,
          Latitude: +lat.value, Longitude: +lon.value
        })
      });
      const j = await res.json();
      h_out.style.display = 'block';
      h_out.textContent = j.result;
    }

    async function predictStudent() {
      const res = await fetch('/predict/student', {
        method: 'POST', headers: {'Content-Type': 'application/json'},
        body: JSON.stringify({
          StudyHours: +sh.value, PreviousScore: +ps.value, SleepHours: +slp.value,
          AttendancePct: +att.value, PracticeTests: +pt.value
        })
      });
      const j = await res.json();
      s_out.style.display = 'block';
      s_out.textContent = j.result;
    }
  </script>
</body>
</html>
'''

@flask_app.route('/')
def home():
    return render_template_string(page)

@flask_app.route('/predict/housing', methods=['POST'])
def pred_housing():
    d = request.get_json()
    x = np.array([[d['MedInc'], d['HouseAge'], d['AveRooms'], d['AveBedrms'],
                   d['Population'], d['AveOccup'], d['Latitude'], d['Longitude']]])
    x = scaler1.transform(x)
    val = model1.predict(x)[0]
    return jsonify({'result': f'House Value: ${val*100000:,.0f}'})

@flask_app.route('/predict/student', methods=['POST'])
def pred_student():
    d = request.get_json()
    x = np.array([[d['StudyHours'], d['PreviousScore'], d['SleepHours'],
                   d['AttendancePct'], d['PracticeTests']]])
    x = scaler2.transform(x)
    val = float(np.clip(model2.predict(x)[0], 0, 100))
    return jsonify({'result': f'Exam Score: {val:.2f} / 100'})


thread = threading.Thread(target=lambda: flask_app.run(port=5050, debug=False, use_reloader=False), daemon=True)
thread.start()

print('Flask running at: http://127.0.0.1:5050')